# Dual-Band Sweep + ORx AGC - ADRV9026

Transmit two bands and sweep a parameter; at **every step** auto-level *each*
ORx independently, then capture both in one aligned snapshot.

## About the ORx "AGC"

There is no trustworthy hardware AGC for the ORx here, so we auto-level in software
on the **captured-IQ peak** (the metric proven faithful on this bench;
`RxDecPowerGet` compresses the range and is not reliable for leveling). At every
sweep point `autolevel_orx` steps the ORx gain index (clamped to the valid
**190..255** window) until the captured peak lands in `ORX_TARGET_DBFS +/- tol`.

It **fails loud**: if the target needs more gain than index 255 (signal too weak)
or less than index 190 (too strong / clipping) it stops and reports a reason rather
than spinning -- watch the `leveled` column and the red X's on the plot.

## 1. Parameters (edit me)

In [ ]:
from adrvtrx import TxChannel, RxChannel

# --- Profile -----------------------------------------------------------------
PROFILES = {
    "98_linksharing":  "ADRV9025Init_StdUseCase98_LinkSharing.profile",   # Tx 491.52 MSPS, Np=12
    "102_linksharing": "ADRV9025Init_StdUseCase102_LinkSharing.profile",  # Np=16
    "14_linksharing":  "ADRV9025Init_StdUseCase14_LinkSharing.profile",
    "50_linksharing":  "ADRV9025Init_StdUseCase50_LinkSharing.profile",
}
PROFILE = "98_linksharing"          # <-- choose here

# --- Bands (bench wiring TX2->ORx2, TX3->ORx3) ------------------------------
INPUT_DIR = "C:/Users/ohammi/OneDrive - aus.edu/DualBand/input_100"
BANDS = [
    {"name": "band1", "tx": TxChannel.TX2, "orx": RxChannel.ORX2, "signal": f"{INPUT_DIR}/Signal1.txt"},
    {"name": "band2", "tx": TxChannel.TX3, "orx": RxChannel.ORX3, "signal": f"{INPUT_DIR}/Signal2.txt"},
]

# --- Sweep -------------------------------------------------------------------
# SWEEP_KIND: "tx_atten" (dB, 0..41.95) or "lo2_freq" (Hz; LO2 drives TX + ORx).
SWEEP_KIND   = "tx_atten"
SWEEP_VALUES = [20.0, 15.0, 10.0, 5.0, 0.0]      # dB for tx_atten
# Frequency-sweep example:
#   SWEEP_KIND = "lo2_freq"; SWEEP_VALUES = [0.8e9, 0.9e9, 1.0e9, 1.1e9]

# --- ORx auto-level (AGC) ----------------------------------------------------
ORX_TARGET_DBFS  = -20.0   # aim point for the captured peak at every sweep step
ORX_TOLERANCE_DB = 2.0
CAPTURE_MS       = 0.05

## 2. Imports, config, profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from adrvtrx.config import load_config
from adrvtrx.profile import read_profile
from adrvtrx.radio import Radio
from adrvtrx.experiment import verify_status
from adrvtrx.waveform import load_tab_iq, full_scale
from adrvtrx.transmit import transmit_bands
from adrvtrx.capture import capture
from adrvtrx.gain import clip_report, autolevel_orx, ORX_GAIN_MIN, ORX_GAIN_MAX
from adrvtrx.sweep import SweepAxis, run_sweep, attenuation_axis, frequency_axis
from adrvtrx._enums import is_orx

cfg = load_config()
cfg.profile_name = PROFILES.get(PROFILE, PROFILE)
info = read_profile(cfg.profile_path)
print("Profile:", cfg.profile_path.name)
print(f"ORx auto-level (AGC) on the captured-IQ peak; valid gain window {ORX_GAIN_MIN}..{ORX_GAIN_MAX}")

## 3. Signal-path summary - sample rates & bit depth

In [ ]:
def _fs(bits):
    return (1 << (bits - 1)) - 1

print(f"{'path':<6}{'rate (MSPS)':>14}{'bits (Np)':>12}{'full scale':>12}")
print(f"{'TX':<6}{info.tx_rate_khz/1000:>14.3f}{info.tx_bits:>12}{_fs(info.tx_bits):>12}")
print(f"{'Rx':<6}{info.rx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")
print(f"{'ORx':<6}{info.orx_rate_khz/1000:>14.3f}{info.rx_bits:>12}{_fs(info.rx_bits):>12}")

## 4. Load the waveforms

In [ ]:
for b in BANDS:
    b["wave"] = load_tab_iq(b["signal"])
    print(f"{b['name']}: {b['tx'].name} -> {b['orx'].name} | {len(b['wave'])} samples")
assert len({len(b["wave"]) for b in BANDS}) == 1, "waveforms must be equal length"

## 5. Connect, program, verify

In [ ]:
radio = Radio(cfg)
radio.connect()
radio.force_safe()
radio.program()
for k, v in verify_status(radio).items():
    print(f"{k}: {v}")

## 6. Build the sweep axis + per-point action (auto-level each ORx, capture both)

In [ ]:
def measure_peak(ch):
    res = capture(radio, int(ch), CAPTURE_MS, bits=info.rx_bits)
    cap = res.channels[ch]
    return clip_report(cap.i, cap.q, info.rx_bits).peak_dbfs

cap_mask = 0
for b in BANDS:
    cap_mask |= int(b["orx"])

def action(point):
    row = dict(point)
    # AGC each band's ORx independently (different ADCs, no interaction).
    for b in BANDS:
        lr = autolevel_orx(
            lambda g, ch=b["orx"]: radio.set_rx_gain(ch, g),
            lambda ch=b["orx"]: measure_peak(ch),
            target_dbfs=ORX_TARGET_DBFS, tolerance_db=ORX_TOLERANCE_DB,
        )
        row[f"{b['name']}_gain"] = lr.final_gain_index
        row[f"{b['name']}_leveled"] = lr.converged
        row[f"{b['name']}_note"] = lr.reason
    # One aligned snapshot of both bands at the settled gains.
    res = capture(radio, cap_mask, CAPTURE_MS, bits=info.rx_bits)
    for b in BANDS:
        cap = res.channels[b["orx"]]
        row[f"{b['name']}_peak"] = clip_report(cap.i, cap.q, info.rx_bits).peak_dbfs
    return row

if SWEEP_KIND == "tx_atten":
    def _set_atten(db):
        for b in BANDS:
            radio.set_tx_atten(b["tx"], float(db))
    axis = SweepAxis("atten_db", _set_atten, SWEEP_VALUES)
    XLABEL, XKEY = "TX atten (dB)", "atten_db"
elif SWEEP_KIND == "lo2_freq":
    axis = frequency_axis(radio, "LO2", [int(v) for v in SWEEP_VALUES], name="lo2_hz")
    XLABEL, XKEY = "LO2 (Hz)", "lo2_hz"
else:
    raise ValueError(f"unknown SWEEP_KIND {SWEEP_KIND!r}")

## 7. Run the sweep

In [ ]:
start_atten = max(SWEEP_VALUES) if SWEEP_KIND == "tx_atten" else 10.0
for b in BANDS:
    radio.set_tx_atten(b["tx"], start_atten)
transmit_bands(radio, {b["tx"]: b["wave"] for b in BANDS}, info.tx_bits,
               continuous=True, do_normalize=True)
records = run_sweep([axis], action)
radio.disable_tx()

hdr = f"{XLABEL:>14}"
for b in BANDS:
    hdr += f" {b['name']+'_gain':>11}{b['name']+'_peak':>11}{b['name']+'_lvl':>7}"
print(hdr)
for r in records:
    line = f"{r[XKEY]:>14}"
    for b in BANDS:
        line += (f" {r[b['name']+'_gain']:>11}{r[b['name']+'_peak']:>11.1f}"
                 f"{str(r[b['name']+'_leveled']):>7}")
    print(line)

## 8. Plot - per-band gain + leveled capture vs sweep

In [ ]:
x = np.array([r[XKEY] for r in records], float)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for b in BANDS:
    ax[0].plot(x, [r[f"{b['name']}_gain"] for r in records], "o-", label=b["name"])
    ax[1].plot(x, [r[f"{b['name']}_peak"] for r in records], "o-", label=b["name"])
ax[1].axhline(ORX_TARGET_DBFS, ls="--", c="k", lw=1, label="target")
ax[0].set_title("settled ORx gain index"); ax[0].set_xlabel(XLABEL)
ax[0].set_ylabel("gain index"); ax[0].legend(); ax[0].grid(True)
ax[1].set_title("captured peak after AGC"); ax[1].set_xlabel(XLABEL)
ax[1].set_ylabel("dBFS"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 9. Safe-state & disconnect

In [ ]:
radio.safe_state()
radio.disconnect()
print("TX safe, board disconnected")